In [ ]:
import os
import requests
import pandas as pd
import time
import streamlit as st

In [ ]:
API_KEY = st.secrets["EIA_API_KEY"]
ENDPOINT_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

# Definimos los bloques temporales (pares de inicio y fin) para traer de 2022 a 2025
bloques_temporales = [
    ("2022-01-01T00", "2022-07-01T00"),
    ("2022-07-01T00", "2023-01-01T00"),
    ("2023-01-01T00", "2023-07-01T00"),
    ("2023-07-01T00", "2024-01-01T00"),
    ("2024-01-01T00", "2024-07-01T00"),
    ("2024-07-01T00", "2025-01-01T00"),
    ("2025-01-01T00", "2025-07-01T00"),
    ("2025-07-01T00", "2026-01-01T00")
]

dataframes_bloques = []

for inicio, fin in bloques_temporales:
    print(f"Descargando bloque: {inicio} al {fin}...")
    
    params = {
        "api_key": API_KEY,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "ERCO",
        "facets[type][]": "D",
        "start": inicio,
        "end": fin,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000 
    }
    
    try:
        response = requests.get(ENDPOINT_URL, params=params)
        response.raise_for_status()
        records = response.json()['response']['data']
        df_bloque = pd.DataFrame(records)
        dataframes_bloques.append(df_bloque)
        
        # Una pequeña pausa de cortesía para no saturar la API
        time.sleep(1)
        
    except Exception as e:
        print(f"Error en el bloque {inicio}-{fin}: {e}")
        break

# Concatenar todos los bloques en un solo gran DataFrame histórico
if dataframes_bloques:
    df_historico_completo = pd.concat(dataframes_bloques, ignore_index=True)
    # Eliminar duplicados si es que se traslaparon las horas de corte
    df_historico_completo = df_historico_completo.drop_duplicates(subset=['period'])
    
    print(f"\n[PROCESO TERMINADO]")
    print(f"Total de registros consolidados: {df_historico_completo.shape[0]}")
    
    # Guardar nuestro dataset maestro de R&D
    df_historico_completo.to_csv('data/ercot_load_2022_2025_static.csv', index=False)
    print("Dataset guardado en: data/ercot_load_2022_2025_static.csv")

Descargando bloque: 2022-01-01T00 al 2022-07-01T00...
Descargando bloque: 2022-07-01T00 al 2023-01-01T00...
Descargando bloque: 2023-01-01T00 al 2023-07-01T00...
Descargando bloque: 2023-07-01T00 al 2024-01-01T00...
Descargando bloque: 2024-01-01T00 al 2024-07-01T00...
Descargando bloque: 2024-07-01T00 al 2025-01-01T00...
Descargando bloque: 2025-01-01T00 al 2025-07-01T00...
Descargando bloque: 2025-07-01T00 al 2026-01-01T00...

[PROCESO TERMINADO]
Total de registros consolidados: 35065
Dataset guardado en: data/ercot_load_2022_2025_static.csv
